# Extraction des interdictions de vente

Extrait depuis le fichier Parquet d’entrée les trois champs d’interdiction (profil Non Retail / Non Natural Persons) pour les zones **EEA, UK, Swiss**, pour les documents classés **Final Terms** ou **Termsheet**.

| Valeur | Signification |
|---|---|
| `True` | Interdiction applicable |
| `False` | Interdiction non applicable |
| `N_A` | Document hors périmètre (Notice, autre type, contenu null…) |
| `Error` | Final Terms / Termsheet : marqueur trouvé mais statut introuvable |

In [ ]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

INPUT    = Path("data_SG.parquet")
OUT_PQ   = Path("resultats.parquet")
OUT_CSV  = Path("resultats.csv")
OUT_XLSX = Path("resultat.xlsx")

In [ ]:
# ── Regex utilitaires ─────────────────────────────────────────────────────────

# Lignes de bruit PyPDF à supprimer (headers / footers / numéros de page)
PAGE_NOISE_RE = re.compile(
    r'^(?:APPLICABLE FINAL TERMS|AMENDED AND RESTATED FINAL TERMS'
    r'|FINAL VERSION APPROVED BY THE ISSUER|CONDITIONS DEFINITIVES APPLICABLES'
    r'|VERSION FINALE APPROUVEE PAR L.EMETTEUR|DEFINITIVES? ENDG[ÜU]LTIGE BEDINGUNGEN'
    r'|VOM EMITTENTEN GENEHMIGTE ENDG[ÜU]LTIGE VERSION|-\s*\d+\s*-|Page\s+\d+\s+of\s+\d+)\s*$',
    re.IGNORECASE
)

# Bornes de la section Distribution / Placement / Platzierung
SEC_START = re.compile(r'(?m)^\s*\d+\s*\.\s*(?:DISTRIBUTION|PLACEMENT|PLATZIERUNG)\b', re.IGNORECASE)
SEC_END   = re.compile(
    r'(?m)^\s*\d+\s*\.\s*(?:TERMS\s+AND\s+CONDITIONS\s+OF\s+THE\s+(?:OFFER|NOTES)'
    r'|MODALIT[ÉE]S\s+DE\s+L.OFFRE|EMISSIONSBEDINGUNGEN\s+DES\s+ANGEBOTS)\b',
    re.IGNORECASE
)

# ── Patterns d'extraction par langue ─────────────────────────────────────────
# Termes "Non Retail / Non Natural" par langue
_NRC_EN = r'(?:Non[\s\-]+Retail\s+(?:Clients|Investors|Persons)|Non[\s\-]+Natural\s+(?:Persons|Clients|Investors))'
_NRC_FR = r'(?:Non\s+Client[eè]le\s+de\s+D[eé]tail|Personnes\s+morales|Investisseurs\s+Qualifi[eé]s)'
_NRC_DE = r'Nicht[\s\-]+Privatpersonen'

# Statuts (groupes capturants)
_S_EN   = r'(Not\s+Applicable|Applicable)'
_S_FR   = r'(Non\s+Applicable|Applicable|Sans\s+[Oo]bjet)'
_S_DE   = r'(Nicht\s+Anwendbar|Anwendbar)'
# Versions sans capture (pour interleaving)
_SI_FR  = r'(?:Non\s+Applicable|Applicable|Sans\s+[Oo]bjet)'
_SI_EN  = r'(?:Not\s+Applicable|Applicable)'
_SI_DE  = r'(?:Nicht\s+Anwendbar|Anwendbar)'

# Bruit DE optionnel intercalé entre label EN et statut (PDF bilingue)
_BILI = r'(?:(?:-\s*|\(\w+\)\s*)?Verkaufsverbot[^:]+:\s*)?'

# Royaume-Uni: tolère espaces/tirets (PDF: "Royaume-Uni" ou "Royaume- Uni")
_RU = r'Royaume[\s\-]+Uni'
# EEE: tolère apostrophe variante
_EEE = r'(?:de\s+l[\u2019\u2018\']|dans\s+l[\u2019\u2018\'])\s*EEE'

def _en(zone, nrc=_NRC_EN):
    return re.compile(
        rf'Prohibition\s+of\s+Sales\s+to\s+{zone}\s+{nrc}\s*:\s*{_BILI}{_S_EN}',
        re.IGNORECASE,
    )

# Plusieurs alternatives par (langue, zone). On prend le premier match.
PATTERNS = {
    'EN': {
        'EEA'  : [_en('EEA')],
        'UK'   : [_en('UK')],
        'Swiss': [_en('Swiss')],
    },
    'FR': {
        # Pattern 1: statut après le ":" (cas standard)
        # Pattern 2: statut interleavé entre "à des" et "Personnes" (PDF 2 colonnes)
        'EEA': [
            re.compile(rf'Interdiction\s+de\s+Ventes[^:]*?{_NRC_FR}\s+{_EEE}\s*:\s*{_S_FR}', re.IGNORECASE),
            re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+{_S_FR}\s+(?:Personnes\s+)?{_NRC_FR}\s+{_EEE}\s*:', re.IGNORECASE),
        ],
        'UK': [
            re.compile(rf'Interdiction\s+de\s+Ventes[^:]*?{_NRC_FR}\s+au\s+{_RU}\s*:\s*{_S_FR}', re.IGNORECASE),
            re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+{_S_FR}\s+(?:Personnes\s+)?{_NRC_FR}\s+au\s+{_RU}\s*:', re.IGNORECASE),
        ],
        'Swiss': [
            re.compile(rf'Interdiction\s+de\s+Ventes[^:]*?{_NRC_FR}\s+en\s+Suisse\s*:\s*{_S_FR}', re.IGNORECASE),
            re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+{_S_FR}\s+(?:Personnes\s+)?{_NRC_FR}\s+en\s+Suisse\s*:', re.IGNORECASE),
        ],
    },
    'DE': {
        'EEA'  : [re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+im\s+EWR\s*:\s*{_S_DE}', re.IGNORECASE)],
        'UK'   : [re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+im\s+Vereinigten\s+K[oö]nigreich\s*:\s*{_S_DE}', re.IGNORECASE)],
        'Swiss': [re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+in\s+der\s+Schweiz\s*:\s*{_S_DE}', re.IGNORECASE)],
    },
}

# Marqueurs "label only" : indiquent si la ligne Non-Retail/Non-Natural existe pour la zone.
# Si aucun marqueur n'est trouvé pour une zone → N_A (champ absent du document).
LABEL_PATTERNS = {
    'EN': {
        'EEA'  : re.compile(rf'Prohibition\s+of\s+Sales\s+to\s+EEA\s+{_NRC_EN}',   re.IGNORECASE),
        'UK'   : re.compile(rf'Prohibition\s+of\s+Sales\s+to\s+UK\s+{_NRC_EN}',    re.IGNORECASE),
        'Swiss': re.compile(rf'Prohibition\s+of\s+Sales\s+to\s+Swiss\s+{_NRC_EN}', re.IGNORECASE),
    },
    'FR': {
        'EEA'  : re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+(?:{_SI_FR}\s+)?(?:Personnes\s+)?{_NRC_FR}\s+{_EEE}',     re.IGNORECASE),
        'UK'   : re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+(?:{_SI_FR}\s+)?(?:Personnes\s+)?{_NRC_FR}\s+au\s+{_RU}', re.IGNORECASE),
        'Swiss': re.compile(rf'Interdiction\s+de\s+Ventes\s+(?:aux|à\s+des)\s+(?:{_SI_FR}\s+)?(?:Personnes\s+)?{_NRC_FR}\s+en\s+Suisse', re.IGNORECASE),
    },
    'DE': {
        'EEA'  : re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+im\s+EWR',                              re.IGNORECASE),
        'UK'   : re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+im\s+Vereinigten\s+K[oö]nigreich',      re.IGNORECASE),
        'Swiss': re.compile(rf'Verkaufsverbot\s+an\s+{_NRC_DE}\s+in\s+der\s+Schweiz',                    re.IGNORECASE),
    },
}

STATUS_MAP = {
    'not applicable': False, 'non applicable': False,
    'sans objet': False,     'nicht anwendbar': False,
    'applicable': True,      'anwendbar': True,
}

# ── Helpers dates & montant ───────────────────────────────────────────────────
_FR_MONTHS = {'janvier':1,'février':2,'fevrier':2,'mars':3,'avril':4,'mai':5,'juin':6,
    'juillet':7,'août':8,'aout':8,'septembre':9,'octobre':10,'novembre':11,'décembre':12,'decembre':12}
_EN_MONTHS = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
    'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}

def _parse_date_text(s, lang):
    parts = s.strip().split()
    if len(parts) != 3: return None
    try:
        d, y = int(parts[0]), int(parts[2])
        m = (_EN_MONTHS if lang == 'EN' else _FR_MONTHS).get(parts[1].lower())
        return datetime(y, m, d).date() if m else None
    except: return None

def _parse_date_dmy(s):
    try: return datetime.strptime(s.strip(), '%d/%m/%Y').date()
    except: return None

def _parse_nominal(s):
    s = re.sub(r'[\s\xa0,]', '', s.strip())
    try: return float(s)
    except: return None

# ── Patterns extraction meta (currency / nominal / dates) ─────────────────────
_APOS  = "[\u2019\u2018'\x92`]"
_D_EN  = r'(\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4})'
_D_FR  = r'(\d{1,2}\s+(?:janvier|f[eé]vrier|mars|avril|mai|juin|juillet|ao[uû]t|septembre|octobre|novembre|d[eé]cembre)\s+\d{4})'
_D_DMY = r'(\d{2}/\d{2}/\d{4})'

META_PATTERNS = {
    'EN': {
        'currency'       : re.compile(r'2\.\s+Specified\s+Currency\s*:\s*([A-Z]{3})'),
        'nominal_agg'    : re.compile(r'in\s+an\s+aggregate\s+(?:principal\s+)?amount\s+of\s+[A-Z]{3}\s+([\d,.]+(?:[\s\xa0]\d+)*)', re.IGNORECASE),
        'nominal_dir'    : re.compile(r'\(i\)\s*[-\u2013]?\s*Tranche\s*:\s*(?:up\s+to\s+)?([A-Z]{3})\s+([\d,.\s\xa0]+\d)', re.IGNORECASE),
        'issue_date_text': re.compile(r'\(i\)\s+Issue\s+Date\s*:\s*' + _D_EN, re.IGNORECASE),
        'issue_date_dmy' : re.compile(r'\(i\)\s+Issue\s+Date\s*:(?:\s*\(DD/MM/YYYY\))?\s*' + _D_DMY, re.IGNORECASE),
        'maturity_text'  : re.compile(r'(?:Maturity|Final\s+Exercise)\s+Date\s*:\s*' + _D_EN, re.IGNORECASE),
        'maturity_dmy'   : re.compile(r'(?:Maturity|Final\s+Exercise)\s+Date\s*:(?:\s*\(DD/MM/YYYY\))?\s*' + _D_DMY, re.IGNORECASE),
    },
    'FR': {
        'currency'       : re.compile(r'2\.\s+Devise\s+Pr[eé]vue\s*:\s*([A-Z]{3})'),
        'nominal_dir'    : re.compile(r'\(i\)\s*[-\u2013]?\s*Tranche\s*:\s*(?:jusqu' + _APOS + r'[àa]\s+)?[A-Z]{3}\s+([\d\s\xa0]+\d)', re.IGNORECASE),
        'issue_date_text': re.compile(r'\(i\)\s+Date\s+d' + _APOS + r'?[Ee][mM]ission\s*:\s*' + _D_FR, re.IGNORECASE),
        'issue_date_dmy' : re.compile(r'\(i\)\s+Date\s+d' + _APOS + r'?[Ee][mM]ission\s*:(?:\s*\(JJ/MM/AAAA\))?\s*' + _D_DMY, re.IGNORECASE),
        'maturity_text'  : re.compile(r'Date\s+d' + _APOS + r'?[ÉEeé]ch[eé]ance\s*:\s*' + _D_FR, re.IGNORECASE),
        'maturity_dmy'   : re.compile(r'Date\s+d' + _APOS + r'?[ÉEeé]ch[eé]ance\s*:(?:\s*\(JJ/MM/AAAA\))?\s*' + _D_DMY, re.IGNORECASE),
    },
}

def _normalize_header(content: str) -> str:
    """Supprime le bruit PyPDF et aplatit les 20 000 premiers chars en une ligne."""
    lines = [l.strip() for l in content[:20000].splitlines()
             if l.strip() and not PAGE_NOISE_RE.match(l.strip())]
    return ' '.join(lines)

def extract_meta(content: str, lang: str) -> dict:
    """Extrait currency, nominal, issue_date, maturity_date depuis le contenu brut."""
    out = {'currency': None, 'nominal': None, 'issue_date': None, 'maturity_date': None}
    pats = META_PATTERNS.get(lang)
    if not pats: return out
    norm = _normalize_header(content)

    m = pats['currency'].search(norm)
    if m: out['currency'] = m.group(1).upper()

    if lang == 'EN':
        m = pats['nominal_agg'].search(norm)
        if m: out['nominal'] = _parse_nominal(m.group(1))
    if out['nominal'] is None:
        m = pats['nominal_dir'].search(norm)
        if m:
            raw = m.group(2) if lang == 'EN' else m.group(1)
            out['nominal'] = _parse_nominal(raw)

    for fname, ptext, pdmy in [('issue_date', 'issue_date_text', 'issue_date_dmy'),
                                ('maturity_date', 'maturity_text', 'maturity_dmy')]:
        m = pats[ptext].search(norm)
        if m:
            out[fname] = _parse_date_text(m.group(1), lang)
        else:
            m = pats[pdmy].search(norm)
            if m: out[fname] = _parse_date_dmy(m.group(1))
    return out

In [ ]:
# ── Pipeline ──────────────────────────────────────────────────────────────────

# Fallback positionnel FR : certains PDF mettent toutes les étiquettes en colonne 1
# puis tous les statuts en colonne 2 (à la fin de la section).
# On apparie alors labels et statuts dans l'ordre.
_FR_LABEL_RE = re.compile(
    r'Interdiction\s+de\s+Ventes\s+(?P<type>aux|à\s+des)\s+(?:Personnes\s+)?'
    r'(?P<term>Non\s+Client[eè]le\s+de\s+D[eé]tail|Investisseurs\s+de\s+D[eé]tail'
    r'|Investisseurs\s+Qualifi[eé]s|Personnes\s+morales)'
    r'\s+(?:de\s+l[\u2019\u2018\']|dans\s+l[\u2019\u2018\']|au\s+|en\s+)\s*'
    r'(?P<zone>EEE|Royaume[\s\-]+Uni|Suisse)',
    re.IGNORECASE,
)
_FR_STATUS_RE = re.compile(r'(Sans\s+[Oo]bjet|Non\s+Applicable|Applicable)', re.IGNORECASE)


def _fr_positional_extract(normalized: str) -> dict:
    """Apparie labels Non-Retail (par ordre) avec statuts trouvés après le dernier label."""
    labels = []
    for m in _FR_LABEL_RE.finditer(normalized):
        zone_raw = m.group('zone').upper()
        if 'EEE' in zone_raw:
            zone = 'EEA'
        elif 'ROYAUME' in zone_raw:
            zone = 'UK'
        elif 'SUISSE' in zone_raw:
            zone = 'Swiss'
        else:
            continue
        term = m.group('term').lower()
        is_non_retail = (
            'investisseurs de d' not in term  # exclut "Investisseurs de Détail"
        )
        labels.append({'end': m.end(), 'zone': zone, 'non_retail': is_non_retail})

    if not labels:
        return {}

    last_end = max(l['end'] for l in labels)
    tail = normalized[last_end:]
    statuses = _FR_STATUS_RE.findall(tail)
    if len(statuses) < len(labels):
        return {}

    out = {}
    for label, raw in zip(labels, statuses):
        if label['non_retail']:
            val = STATUS_MAP.get(raw.lower().strip())
            if val is not None:
                out[label['zone']] = val
    return out


def classify_doc(content) -> str:
    """FinalTerms | Termsheet | Notice | NULL | Other."""
    # Gère proprement `pd.NA` (NAType), NaN et None
    if content is None or pd.isna(content):
        return 'NULL'
    if not isinstance(content, str):
        return 'NULL'

    if any(k in content for k in [
        'APPLICABLE FINAL TERMS', 'AMENDED AND RESTATED FINAL TERMS',
        'CONDITIONS DEFINITIVES APPLICABLES',
        'DEFINITIVE ENDGÜLTIGE BEDINGUNGEN', 'DEFINITIVE ENDGULTIGE BEDINGUNGEN',
    ]):
        return 'FinalTerms'
    c = content.strip()
    if c.startswith(('Termsheet', 'Indicative Termsheet', 'Fixed Termsheet')):
        return 'Termsheet'
    if any(k in content for k in [
        'NOTICE TO THE NOTEHOLDER', 'NOTICE TO THE CERTIFICATEHOLDER',
        'Notice to the Certificateholders', 'Notice to Certificateholders',
    ]):
        return 'Notice'
    return 'Other'


def _infer_pdf_name(row: pd.Series) -> str:
    # Compat: certains inputs ont `pdf_name` (extrac.parquet), d'autres `fileName` / `pdf_local_path` (data_SG.parquet)
    v = row.get('pdf_name')
    if isinstance(v, str) and v:
        return v

    v = row.get('fileName')
    if isinstance(v, str) and v:
        return v

    v = row.get('pdf_local_path')
    if isinstance(v, str) and v:
        # garde uniquement le nom du fichier
        return v.rsplit('/', 1)[-1]

    return ''


def process_row(row: pd.Series) -> dict:
    out = {
        'isin'             : row['isin'],
        'docid'            : row['docid'],
        'pdf_name'         : _infer_pdf_name(row),
        'url'              : row.get('url', ''),
        'doc_type'         : '',
        'language'         : '',
        'currency'         : None,
        'nominal'          : None,
        'issue_date'       : None,
        'maturity_date'    : None,
        'prohibition_EEA'  : 'Error',
        'prohibition_UK'   : 'Error',
        'prohibition_Swiss': 'Error',
        'section_text'     : '',
    }
    content  = row.get('content')
    doc_type = classify_doc(content)
    out['doc_type'] = doc_type

    if doc_type not in ('FinalTerms', 'Termsheet'):
        out['prohibition_EEA'] = out['prohibition_UK'] = out['prohibition_Swiss'] = 'N_A'
        return out

    # ── Méta-données (currency / nominal / dates) ─────────────────────────────
    lang_raw = ('EN' if 'Specified Currency' in content
                else 'FR' if 'Devise Pr' in content
                else None)
    if lang_raw:
        meta = extract_meta(content, lang_raw)
        out.update(meta)

    # ── Section Distribution / Placement ─────────────────────────────────────
    m = SEC_START.search(content)
    if not m:
        # Aucune section Distribution: si aucun marqueur Non-Retail nulle part dans
        # le document → N_A pour les 3 zones (cas Final Terms / Termsheet anciens).
        any_label = any(
            lp.search(content)
            for lps in LABEL_PATTERNS.values()
            for lp in lps.values()
        )
        if not any_label:
            out['prohibition_EEA'] = out['prohibition_UK'] = out['prohibition_Swiss'] = 'N_A'
        return out
    e = SEC_END.search(content, m.end())
    section = content[m.start(): e.start() if e else m.start() + 4000]

    # Nettoyer le bruit PyPDF et aplatir en une seule ligne
    normalized = ' '.join(
        l.strip() for l in section.splitlines()
        if l.strip() and not PAGE_NOISE_RE.match(l.strip())
    )
    out['section_text'] = normalized

    # ── Langue(s) détectée(s) (un document peut être bilingue EN/DE) ─────────
    detected = []
    if 'Prohibition of Sales' in normalized:   detected.append('EN')
    if 'Interdiction de Ventes' in normalized: detected.append('FR')
    if 'Verkaufsverbot' in normalized:         detected.append('DE')
    out['language'] = '+'.join(detected) if detected else 'Unknown'

    # ── Interdictions ─────────────────────────────────────────────────────────
    # Pour chaque zone, on cherche dans toutes les langues détectées:
    #  1) si AUCUN label Non-Retail/Non-Natural n'existe pour la zone → N_A
    #  2) sinon, on essaie chaque pattern jusqu'à match → True/False
    #  3) si label trouvé mais aucun statut extractible → 'Error'
    fr_fallback = _fr_positional_extract(normalized) if 'FR' in detected else {}

    for zone in ('EEA', 'UK', 'Swiss'):
        # 1) Détection N_A : aucun label dans aucune langue détectée
        if detected:
            label_found = any(
                LABEL_PATTERNS[l][zone].search(normalized) for l in detected
            )
        else:
            label_found = False
        if not label_found:
            out[f'prohibition_{zone}'] = 'N_A'
            continue

        # 2) Tentative d'extraction du statut
        status = None
        for l in detected:
            for pat in PATTERNS.get(l, {}).get(zone, []):
                hit = pat.search(normalized)
                if hit:
                    raw = next((g for g in hit.groups() if g), None)
                    if raw is not None:
                        status = STATUS_MAP.get(raw.lower().strip())
                        break
            if status is not None:
                break

        # 3) Fallback positionnel FR (PDF mettant les statuts en bloc à la fin)
        if status is None and zone in fr_fallback:
            status = fr_fallback[zone]

        out[f'prohibition_{zone}'] = status if status is not None else 'Error'
    return out

In [ ]:
# ── Traitement & export ───────────────────────────────────────────────────────

df_src = pd.read_parquet(INPUT)

# Garder uniquement les lignes avec un contenu non vide
m = df_src['content'].notna() & df_src['content'].astype('string').str.strip().ne('')
df_src = df_src.loc[m].copy()

print(f"{len(df_src)} documents chargés (content non vide)")

results = [process_row(row) for _, row in tqdm(df_src.iterrows(), total=len(df_src), unit='doc')]

df = pd.DataFrame(results)
PROB = ['prohibition_EEA', 'prohibition_UK', 'prohibition_Swiss']

# Convertir bool → str pour compatibilité Parquet (les valeurs True/False sont stockées en string)
for col in PROB:
    df[col] = df[col].apply(lambda x: str(x) if isinstance(x, bool) else x)

# Typage des colonnes meta
df['nominal']       = pd.to_numeric(df['nominal'], errors='coerce')
df['issue_date']    = pd.to_datetime(df['issue_date'], errors='coerce').dt.date
df['maturity_date'] = pd.to_datetime(df['maturity_date'], errors='coerce').dt.date

df.to_parquet(OUT_PQ, index=False)
df.drop(columns=['section_text']).to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

# Export Excel
with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    df.drop(columns=['section_text']).to_excel(writer, index=False, sheet_name='resultats')

# ── Résumé ────────────────────────────────────────────────────────────────────
EXTRACT_TYPES = ('FinalTerms', 'Termsheet')
extracted = df[df['doc_type'].isin(EXTRACT_TYPES)]
errors = extracted[PROB].apply(lambda r: r.eq('Error').any(), axis=1)

print(f"\nTypes de document :\n{df['doc_type'].value_counts().to_string()}")
print(f"\nFinal Terms + Termsheet : {len(extracted)} docs (périmètre extraction)")
n_ext = len(extracted)
if n_ext:
    print(f"  Succès  : {(~errors).sum()} ({100 * (~errors).sum() / n_ext:.1f}%)")
    print(f"  Erreurs : {errors.sum()}")
else:
    print("  (aucun document dans le périmètre)")
print(f"\nDistribution des valeurs (interdictions) :")
for col in PROB:
    print(f"  {col}: {extracted[col].value_counts(dropna=False).to_dict()}")

print(f"\nChamps méta (Final Terms / Termsheet, si patterns reconnus) :")
print(f"  currency   : {extracted['currency'].notna().sum()}/{n_ext} extraits")
print(f"  nominal    : {extracted['nominal'].notna().sum()}/{n_ext} extraits")
print(f"  issue_date : {extracted['issue_date'].notna().sum()}/{n_ext} extraits")
print(f"  maturity   : {extracted['maturity_date'].notna().sum()}/{n_ext} extraits")

In [ ]:
# ── Aperçu du résultat ────────────────────────────────────────────────────────
display(
    df.drop(columns=['section_text'])
      .sort_values(['doc_type', 'isin'])
      .reset_index(drop=True)
)

In [ ]:
# Lignes avec au moins un champ à True
PROB = ['prohibition_EEA', 'prohibition_UK', 'prohibition_Swiss']

df_true = (
    df[df[PROB].eq('True').any(axis=1)]
    .drop(columns=['section_text'])
    .reset_index(drop=True)
    .copy()
)
print(f"{len(df_true)} lignes avec au moins un True")
display(df_true)

In [ ]:
df_true.iloc[21]["url"]